In [4]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
from numpy import genfromtxt

import json
from math import log10, floor

In [11]:
import json
from math import floor, log10
from partial_participation_eval import load_client_last_summary

fed_m = ["FedAvg"]  ## "FedAdam", "FedAdagrad", "FedYogi", "FedProx"
unlearn_m = ["GradAscent", "GradDiff", "NPO", "SimNPO"]  # , "RMU"

# significant digit
r = 3

rd = 1
rd1 = 5
rd2 = 5
c = 0

def round_sig(x, sig):
    if x == 0 or str(x) == "nan":
        return 0
    return round(x, sig - int(floor(log10(abs(x)))) - 1)

def get_client_metrics(run_dir, client_idx, max_round):
    round_id, summary = load_client_last_summary(
        run_dir=run_dir,
        client=client_idx,
        max_round=max_round,
        stage="post_train",
        summary_file="TOFU_SUMMARY.json",
    )
    if summary is None:
        return None, None, None
    return round_id, summary.get("model_utility"), summary.get("forget_truth_ratio")

for f in fed_m:
    print(f, end=" & ")

    full_dir = f"./saves/TOFU/fedperm_only/full_{f}_Llama-3.2-1B-Instruct-lora_retain90"
    used_round, mu, ftr = get_client_metrics(full_dir, c, rd1)
    if mu is None or ftr is None:
        print("", end=" & ")
        print("", end=" & ")
    else:
        print(round_sig(mu, r), end=" & ")
        print(round_sig(ftr, r), end=" & ")
        # 如果你想看实际用了哪一轮，可以取消下一行注释
        # print(f"[full client{c} uses round {used_round}]", end=" ")

    for u in unlearn_m:
        unlearn_dir = (
            f"./saves/TOFU/fedperm_only/"
            f"unlearn_{f}_Llama-3.2-1B-Instruct-lora_retain90_{u}_filter1_CE1_Alpha1_Base1"
        )
        used_round, mu, ftr = get_client_metrics(unlearn_dir, c, rd)
        if mu is None or ftr is None:
            print("", end=" & ")
            print("", end=" & ")
        else:
            print(round_sig(mu, r), end=" & ")
            print(round_sig(ftr, r), end=" & ")

    retain_dir = f"./saves/TOFU/fedperm_only/retain_{f}_Llama-3.2-1B-Instruct-lora_retain90"
    used_round, mu, ftr = get_client_metrics(retain_dir, c, rd2)
    if mu is None or ftr is None:
        print("", end=" & \\\\\n")
    else:
        print(round_sig(mu, r), end=" & ")
        print(round_sig(ftr, r), end=" \\\\\n")

FedAvg & 0.418 & 0.717 & 0.418 & 0.717 & 0.418 & 0.717 & 0.418 & 0.717 & 0.418 & 0.717 & 0.429 & 0.73 \\


In [2]:
fed_m = ["FedAvg"]  ## "FedAdam", "FedAdagrad", "FedYogi", "FedProx"
# f = "FedAvg"
unlearn_m = ["GradAscent", "GradDiff", "NPO", "SimNPO"]#, "RMU"]
# u = "GradAscent"

# significant digit
r = 3

rd = 1
rd1 = 5
rd2 = 5
c = 9

def round_sig(x, sig):
    if x == 0 or str(x) == 'nan':
        return 0
    return round(x, sig-int(floor(log10(abs(x))))-1)

for f in fed_m:
    print(f, end=' & ')
    try:
        with open(f"./saves/TOFU/fedperm_only/full_{f}_Llama-3.2-1B-Instruct-lora_retain90/eval_rounds/round_{rd1}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:  # client{c}_post_train  server_post_agg
            js = json.loads(j.read())
        # print(round_sig(js['forget_quality'], r), end=' & ')
        print(round_sig(js['model_utility'], r), end=' & ')
        print(round_sig(js['forget_truth_ratio'], r), end=' & ')
    except Exception:
        print("", end=' & ')
        print("", end=' & ')
    for u in unlearn_m:
        try:
            with open(f"./saves/TOFU/fedperm_only/unlearn_{f}_Llama-3.2-1B-Instruct-lora_retain90_{u}_filter1_CE1_Alpha1_Base1/eval_rounds/round_{rd}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
                js = json.loads(j.read())
            # print(round_sig(js['forget_quality'], r), end=' & ')
            print(round_sig(js['model_utility'], r), end=' & ')
            print(round_sig(js['forget_truth_ratio'], r), end=' & ')
        except Exception:
            print("", end=' & ')
            print("", end=' & ')
    try:
        with open(f"./saves/TOFU/fedperm_only/retain_{f}_Llama-3.2-1B-Instruct-lora_retain90/eval_rounds/round_{rd2}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
            js = json.loads(j.read())
        # print(round_sig(js['forget_quality'], r), end=' & ')
        print(round_sig(js['model_utility'], r), end=' & ')
        print(round_sig(js['forget_truth_ratio'], r), end=' \\\\\n')
    except Exception:
        print("", end=' & \\\\\n')
    

FedAvg &  &  &  &  &  &  &  &  &  &  &  & \\


In [25]:
import json
from math import floor, log10
from partial_participation_eval import aggregate_client_last_summaries

fed_m = ["FedAvg"]  ## "FedAdam", "FedAdagrad", "FedYogi", "FedProx"
unlearn_m = ["GradAscent", "GradDiff", "NPO", "SimNPO"]  # , "RMU"

# significant digit
r = 3

# 在这些轮次之前，按“每个客户端最后一次参与的轮次”取结果
rd = 1    # unlearn
rd1 = 5   # full
rd2 = 5   # retain

# 要平均的客户端
c_list = [1]

def round_sig(x, sig):
    if x is None or x == 0 or str(x) == "nan":
        return 0
    return round(x, sig - int(floor(log10(abs(x)))) - 1)

def get_avg_result(run_dir, clients, max_round):
    res = aggregate_client_last_summaries(
        run_dir=run_dir,
        clients=clients,
        metrics=["model_utility", "forget_truth_ratio"],
        max_round=max_round,
        stage="post_train",
        summary_file="TOFU_SUMMARY.json",
    )
    mu = res["average"]["model_utility"]
    ftr = res["average"]["forget_truth_ratio"]
    return mu, ftr, res["per_client"]

for f in fed_m:
    print(f, end=" & ")

    full_dir = f"./saves/TOFU/fedperm_only/full_{f}_Llama-3.2-1B-Instruct-lora_retain90"
    mu, ftr, per_client = get_avg_result(full_dir, c_list, rd1)
    if mu is None or ftr is None:
        print("", end=" & ")
        print("", end=" & ")
    else:
        print(round_sig(mu, r), end=" & ")
        print(round_sig(ftr, r), end=" & ")
        # print("full:", {k: v["round"] for k, v in per_client.items()})

    for u in unlearn_m:
        unlearn_dir = (
            f"./saves/TOFU/fedperm_only/"
            f"unlearn_{f}_Llama-3.2-1B-Instruct-lora_retain90_{u}_filter1_CE1_Alpha1_Base1"
        )
        mu, ftr, per_client = get_avg_result(unlearn_dir, c_list, rd)
        if mu is None or ftr is None:
            print("", end=" & ")
            print("", end=" & ")
        else:
            print(round_sig(mu, r), end=" & ")
            print(round_sig(ftr, r), end=" & ")
            # print(u, {k: v["round"] for k, v in per_client.items()})

    retain_dir = f"./saves/TOFU/fedperm_only/retain_{f}_Llama-3.2-1B-Instruct-lora_retain90"
    mu, ftr, per_client = get_avg_result(retain_dir, c_list, rd2)
    if mu is None or ftr is None:
        print("", end=" & \\\\\n")
    else:
        print(round_sig(mu, r), end=" & ")
        print(round_sig(ftr, r), end=" \\\\\n")
        # print("retain:", {k: v["round"] for k, v in per_client.items()})

FedAvg & 0.422 & 0.73 &  &  &  &  &  &  &  &  & 0.425 & 0.732 \\


In [6]:
fed_m = ["FedAvg", "FedAvgM", "FedAdam", "FedAdagrad", "FedYogi", "FedProx"]
# f = "FedAvg"
unlearn_m = ["GradAscent", "GradDiff", "NPO", "SimNPO"]#, "RMU"]
# u = "GradAscent"

# significant digit
r = 3

rd = 1
rd1 = 11
rd2 = 11
c = 1

def round_sig(x, sig):
    if x == 0 or str(x) == 'nan':
        return 0
    return round(x, sig-int(floor(log10(abs(x))))-1)

for f in fed_m:
    print(f, end=' & ')
    try:
        with open(f"./saves/TOFU/fedperm_only/full_{f}_Qwen2.5-1.5B-Instruct-lora_retain90/eval_rounds/round_{rd1}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
            js = json.loads(j.read())
        # print(round_sig(js['forget_quality'], r), end=' & ')
        print(round_sig(js['model_utility'], r), end=' & ')
        print(round_sig(js['forget_truth_ratio'], r), end=' & ')
    except Exception:
        print("", end=' & ')
        print("", end=' & ')
    for u in unlearn_m:
        try:
            with open(f"./saves/TOFU/fedperm_only/unlearn_{f}_Qwen2.5-1.5B-Instruct-lora_retain90_{u}_filter1_CE1_Alpha1/eval_rounds/round_{rd}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
                js = json.loads(j.read())
            # print(round_sig(js['forget_quality'], r), end=' & ')
            print(round_sig(js['model_utility'], r), end=' & ')
            print(round_sig(js['forget_truth_ratio'], r), end=' & ')
        except Exception:
            print("", end=' & ')
            print("", end=' & ')
    try:
        with open(f"./saves/TOFU/fedperm_only/retain_{f}_Qwen2.5-1.5B-Instruct-lora_retain90/eval_rounds/round_{rd2}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
            js = json.loads(j.read())
        # print(round_sig(js['forget_quality'], r), end=' & ')
        print(round_sig(js['model_utility'], r), end=' & ')
        print(round_sig(js['forget_truth_ratio'], r), end=' \\\\\n')
    except Exception:
        print("", end=' & \\\\\n')
    

FedAvg & 0.507 & 0.584 & 0.458 & 0.689 & 0.458 & 0.689 & 0.455 & 0.667 & 0.467 & 0.652 & 0.506 & 0.685 \\
FedAvgM & 0.496 & 0.453 & 0.423 & 0.576 & 0.423 & 0.576 & 0.432 & 0.557 & 0.437 & 0.545 & 0.487 & 0.612 \\
FedAdam & 0.422 & 0.701 & 0.414 & 0.716 & 0.414 & 0.716 & 0.425 & 0.707 & 0.43 & 0.689 & 0.421 & 0.72 \\
FedAdagrad & 0.462 & 0.694 & 0.45 & 0.725 & 0.45 & 0.725 & 0.441 & 0.717 & 0.445 & 0.688 & 0.506 & 0.685 \\
FedYogi & 0.423 & 0.702 & 0.419 & 0.716 & 0.419 & 0.716 & 0.426 & 0.708 & 0.43 & 0.689 & 0.421 & 0.72 \\
FedProx & 0.495 & 0.615 & 0.461 & 0.692 & 0.461 & 0.692 & 0.455 & 0.676 & 0.456 & 0.656 & 0.496 & 0.69 \\


In [4]:
fed_m = ["FedAvg", "FedAvgM", "FedAdam", "FedAdagrad", "FedYogi", "FedProx"]
# f = "FedAvg"
unlearn_m = ["GradAscent", "GradDiff", "NPO", "SimNPO"]#, "RMU"]
# u = "GradAscent"
m = "ra_Q_A_ROUGE"
# significant digit
r = 3
# TODO: ra_Q_A_ROUGE, wf_Q_A_ROUGE, forget_Q_A_ROUGE, retain_Q_A_ROUGE
rd = 1
rd1 = 11
rd2 = 11
c = 1

def round_sig(x, sig):
    if x == 0 or str(x) == 'nan':
        return 0
    return round(x, sig-int(floor(log10(abs(x))))-1)

for f in fed_m:
    print(f, end=' & ')
    try:
        with open(f"./saves/TOFU/fedperm_only/full_{f}_Llama-3.2-1B-Instruct-lora_retain90/eval_rounds/round_{rd1}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
            js = json.loads(j.read())
        # print(round_sig(js['forget_quality'], r), end=' & ')
        print(round_sig(js[m], r), end=' & ')
    except Exception:
        print("", end=' & ')
    for u in unlearn_m:
        try:
            with open(f"./saves/TOFU/fedperm_only/unlearn_{f}_Llama-3.2-1B-Instruct-lora_retain90_{u}_filter1_CE1_Alpha1/eval_rounds/round_{rd}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
                js = json.loads(j.read())
            # print(round_sig(js['forget_quality'], r), end=' & ')
            print(round_sig(js[m], r), end=' & ')
        except Exception:
            print("", end=' & ')
    try:
        with open(f"./saves/TOFU/fedperm_only/retain_{f}_Llama-3.2-1B-Instruct-lora_retain90/eval_rounds/round_{rd2}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
            js = json.loads(j.read())
        # print(round_sig(js['forget_quality'], r), end=' & ')
        print(round_sig(js[m], r), end=' \\\\\n')
    except Exception:
        print("", end=' & \\\\\n')
    

FedAvg & 0.717 & 0.8 & 0.797 & 0.794 & 0.782 & 0.704 \\
FedAvgM & 0.723 & 0.794 & 0.812 & 0.778 & 0.794 & 0.458 \\
FedAdam & 0.723 & 0.794 & 0.802 & 0.778 & 0.794 & 0.84 \\
FedAdagrad & 0.728 & 0.795 & 0.784 & 0.768 & 0.773 & 0.83 \\
FedYogi & 0.723 & 0.794 & 0.812 & 0.778 & 0.794 & 0.787 \\
FedProx & 0.741 & 0.816 & 0.816 & 0.823 & 0.775 & 0.731 \\


In [9]:
fed_m = ["FedAvg", "FedAvgM", "FedAdam", "FedAdagrad", "FedYogi", "FedProx"]
# f = "FedAvg"
unlearn_m = ["GradAscent", "GradDiff", "NPO", "SimNPO"]#, "RMU"]
# u = "GradAscent"

# significant digit
r = 3

rd = 1
rd1 = 11
rd2 = 11
c = 1

def round_sig(x, sig):
    if x == 0 or str(x) == 'nan':
        return 0
    return round(x, sig-int(floor(log10(abs(x))))-1)

for f in fed_m:
    print(f, end=' & ')
    try:
        with open(f"./saves/TOFU/fedperm_only/full_{f}_Qwen2.5-1.5B-Instruct-lora_retain90/eval_rounds/round_{rd1}/server_post_agg/TOFU_SUMMARY.json", 'r') as j:
            js = json.loads(j.read())
        # print(round_sig(js['forget_quality'], r), end=' & ')
        print(round_sig(js['model_utility'], r), end=' & ')
        print(round_sig(js['forget_truth_ratio'], r), end=' & ')
    except Exception:
        print("", end=' & ')
        print("", end=' & ')
    for u in unlearn_m:
        try:
            with open(f"./saves/TOFU/fedperm_only/unlearn_{f}_Qwen2.5-1.5B-Instruct-lora_retain90_{u}_filter1_CE1_Alpha1/eval_rounds/round_{rd}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
                js = json.loads(j.read())
            # print(round_sig(js['forget_quality'], r), end=' & ')
            print(round_sig(js['model_utility'], r), end=' & ')
            print(round_sig(js['forget_truth_ratio'], r), end=' & ')
        except Exception:
            print("", end=' & ')
            print("", end=' & ')
    try:
        with open(f"./saves/TOFU/fedperm_only/retain_{f}_Qwen2.5-1.5B-Instruct-lora_retain90/eval_rounds/round_{rd2}/server_post_agg/TOFU_SUMMARY.json", 'r') as j:
            js = json.loads(j.read())
        # print(round_sig(js['forget_quality'], r), end=' & ')
        print(round_sig(js['model_utility'], r), end=' & ')
        print(round_sig(js['forget_truth_ratio'], r), end=' \\\\\n')
    except Exception:
        print("", end=' & \\\\\n')
    

FedAvg & 0.532 & 0.55 & 0.458 & 0.689 & 0.458 & 0.689 & 0.455 & 0.667 & 0.467 & 0.652 & 0.522 & 0.681 \\
FedAvgM & 0.504 & 0.446 & 0.423 & 0.576 & 0.423 & 0.576 & 0.432 & 0.557 & 0.437 & 0.545 & 0.411 & 0.61 \\
FedAdam & 0.237 & 0.712 & 0.414 & 0.716 & 0.414 & 0.716 & 0.425 & 0.707 & 0.43 & 0.689 & 0.318 & 0.726 \\
FedAdagrad & 0.465 & 0.689 & 0.45 & 0.725 & 0.45 & 0.725 & 0.441 & 0.717 & 0.445 & 0.688 & 0.522 & 0.681 \\
FedYogi & 0.241 & 0.712 & 0.419 & 0.716 & 0.419 & 0.716 & 0.426 & 0.708 & 0.43 & 0.689 & 0.308 & 0.727 \\
FedProx & 0.512 & 0.587 & 0.461 & 0.692 & 0.461 & 0.692 & 0.455 & 0.676 & 0.456 & 0.656 & 0.517 & 0.68 \\


In [5]:
fed_m = ["FedAvg", "FedAvgM", "FedAdam", "FedAdagrad", "FedYogi", "FedProx"]
# f = "FedAvg"
unlearn_m = ["GradAscent", "GradDiff", "NPO", "SimNPO"]#, "RMU"]
# u = "GradAscent"

# significant digit
r = 3

rd = 5
rd1 = 15
rd2 = 15
c = 1

def round_sig(x, sig):
    if x == 0 or str(x) == 'nan':
        return 0
    return round(x, sig-int(floor(log10(abs(x))))-1)

for f in fed_m:
    print(f, end=' & ')
    try:
        with open(f"./saves/TOFU/fedperm_only/full_{f}_Qwen2.5-1.5B-Instruct-lora_retain90/eval_rounds/round_{rd1}/server_post_agg/TOFU_SUMMARY.json", 'r') as j:
            js = json.loads(j.read())
        # print(round_sig(js['forget_quality'], r), end=' & ')
        print(round_sig(js['model_utility'], r), end=' & ')
        print(round_sig(js['forget_truth_ratio'], r), end=' & ')
    except Exception:
        print("", end=' & ')
        print("", end=' & ')
    for u in unlearn_m:
        try:
            with open(f"./saves/TOFU/fedperm_only/unlearn_{f}_Qwen2.5-1.5B-Instruct-lora_retain90_{u}_filter1_CE1_Alpha1/eval_rounds/round_{rd}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
                js = json.loads(j.read())
            # print(round_sig(js['forget_quality'], r), end=' & ')
            print(round_sig(js['model_utility'], r), end=' & ')
            print(round_sig(js['forget_truth_ratio'], r), end=' & ')
        except Exception:
            print("", end=' & ')
            print("", end=' & ')
    try:
        with open(f"./saves/TOFU/fedperm_only/retain_{f}_Qwen2.5-1.5B-Instruct-lora_retain90/eval_rounds/round_{rd2}/server_post_agg/TOFU_SUMMARY.json", 'r') as j:
            js = json.loads(j.read())
        # print(round_sig(js['forget_quality'], r), end=' & ')
        print(round_sig(js['model_utility'], r), end=' & ')
        print(round_sig(js['forget_truth_ratio'], r), end=' \\\\\n')
    except Exception:
        print("", end=' & \\\\\n')
    

FedAvg & 0.557 & 0.512 & 0.525 & 0.606 & 0.5 & 0.611 & 0.531 & 0.585 & 0.522 & 0.571 & 0.536 & 0.663 \\
FedAvgM & 0.494 & 0.428 & 0.494 & 0.446 & 0.494 & 0.446 & 0.478 & 0.452 & 0.491 & 0.446 & 0.44 & 0.587 \\
FedAdam & 0.385 & 0.699 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0.41 & 0.717 \\
FedAdagrad & 0.487 & 0.646 & 8.59e-05 & 0.737 & 8.59e-05 & 0.737 & 0 & 0.725 & 7.47e-05 & 0.727 & 0.536 & 0.663 \\
FedYogi & 0.385 & 0.7 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0.408 & 0.718 \\
FedProx & 0.53 & 0.556 & 0.51 & 0.603 & 0.51 & 0.603 & 0.511 & 0.59 & 0.515 & 0.568 & 0.545 & 0.656 \\


In [6]:
fed_m = ["FedAvg", "FedAvgM", "FedAdam", "FedAdagrad", "FedYogi", "FedProx"]
# f = "FedAvg"
unlearn_m = ["GradAscent", "GradDiff", "NPO", "SimNPO"]#, "RMU"]
# u = "GradAscent"

# significant digit
r = 3

rd = 1
rd1 = 10
rd2 = 11
c = 1

def round_sig(x, sig):
    if x == 0 or str(x) == 'nan':
        return 0
    return round(x, sig-int(floor(log10(abs(x))))-1)

for f in fed_m:
    print(f, end=' & ')
    try:
        with open(f"./saves/TOFU/fedperm_only/full_{f}_Llama-3.1-8B-Instruct-lora_retain90/eval_rounds/round_{rd1}/server_post_agg/TOFU_SUMMARY.json", 'r') as j:
            js = json.loads(j.read())
        # print(round_sig(js['forget_quality'], r), end=' & ')
        print(round_sig(js['model_utility'], r), end=' & ')
        print(round_sig(js['forget_truth_ratio'], r), end=' & ')
    except Exception:
        print("", end=' & ')
        print("", end=' & ')
    for u in unlearn_m:
        try:
            with open(f"./saves/TOFU/fedperm_only/unlearn_{f}_Llama-3.1-8B-Instruct-lora_retain90_{u}_filter1_CE1_Alpha1/eval_rounds/round_{rd}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
                js = json.loads(j.read())
            # print(round_sig(js['forget_quality'], r), end=' & ')
            print(round_sig(js['model_utility'], r), end=' & ')
            print(round_sig(js['forget_truth_ratio'], r), end=' & ')
        except Exception:
            print("", end=' & ')
            print("", end=' & ')
    try:
        with open(f"./saves/TOFU/fedperm_only/retain_{f}_Llama-3.1-8B-Instruct-lora_retain90/eval_rounds/round_{rd2}/server_post_agg/TOFU_SUMMARY.json", 'r') as j:
            js = json.loads(j.read())
        # print(round_sig(js['forget_quality'], r), end=' & ')
        print(round_sig(js['model_utility'], r), end=' & ')
        print(round_sig(js['forget_truth_ratio'], r), end=' \\\\\n')
    except Exception:
        print("", end=' & \\\\\n')
    

FedAvg & 0.698 & 0.49 & 0.592 & 0.658 &  &  &  &  & 0.603 & 0.612 & 0.687 & 0.67 \\
FedAvgM & 0.585 & 0.509 &  &  &  &  &  &  & 0.597 & 0.546 &  & \\
FedAdam &  &  &  &  &  &  &  &  &  &  &  & \\
FedAdagrad &  &  &  &  &  &  &  &  &  &  &  & \\
FedYogi &  &  &  &  &  &  &  &  &  &  &  & \\
FedProx &  &  &  &  &  &  &  &  &  &  & 0.687 & 0.665 \\


In [7]:
f = "FedAvg"
u = "GradAscent"
u2 = "SimNPO"

models = ["Qwen2.5-1.5B", "Qwen2.5-3B", "Qwen2.5-7B", "Llama-3.2-1B", "Llama-3.2-3B", "Llama-3.1-8B"]
split_ratios = ["90", "95", "99"]

# significant digit
r = 3

rd = 5
rd2 = 15
c = 1

def round_sig(x, sig):
    if x == 0:
        return 0
    return round(x, sig-int(floor(log10(abs(x))))-1)

for m in models:
    print(m, end=' & ')

print("", end='\\\\\n FT & ')

for rt in split_ratios:
    for m in models:
        try:
            with open(f"./saves/TOFU/fedperm_only/full_{f}_{m}-Instruct-lora_retain{rt}/eval_rounds/round_{rd2}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
                js = json.loads(j.read())
            print(round_sig(js['model_utility'], r), end=' & ')
        except Exception:
            print("", end=' & ')

    print("", end='\\\\\n FT & ')

    for m in models:
        try:
            with open(f"./saves/TOFU/fedperm_only/full_{f}_{m}-Instruct-lora_retain{rt}/eval_rounds/round_{rd2}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
                js = json.loads(j.read())
            print(round_sig(js['forget_truth_ratio'], r), end=' & ')
        except Exception:
            print("", end=' & ')

    # print("", end='\\\\\n GA & ')

    # for m in models:
    #     try:
    #         with open(f"./saves/TOFU/fedperm_only/unlearn_{f}_{m}-Instruct-lora_retain{rt}_{u}_filter1_CE1_Alpha1/eval_rounds/round_{rd}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
    #             js = json.loads(j.read())
    #         print(round_sig(js['model_utility'], r), end=' & ')
    #     except Exception:
    #         print("", end=' & ')

    # print("", end='\\\\\n GA & ')

    # for m in models:
    #     try:
    #         with open(f"./saves/TOFU/fedperm_only/unlearn_{f}_{m}-Instruct-lora_retain{rt}_{u}_filter1_CE1_Alpha1/eval_rounds/round_{rd}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
    #             js = json.loads(j.read())
    #         print(round_sig(js['forget_truth_ratio'], r), end=' & ')
    #     except Exception:
    #         print("", end=' & ')

    print("", end='\\\\\n SN & ')

    for m in models:
        try:
            with open(f"./saves/TOFU/fedperm_only/unlearn_{f}_{m}-Instruct-lora_retain{rt}_{u2}_filter1_CE1_Alpha1/eval_rounds/round_{rd}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
                js = json.loads(j.read())
            print(round_sig(js['model_utility'], r), end=' & ')
        except Exception:
            print("", end=' & ')

    print("", end='\\\\\n SN & ')

    for m in models:
        try:
            with open(f"./saves/TOFU/fedperm_only/unlearn_{f}_{m}-Instruct-lora_retain{rt}_{u2}_filter1_CE1_Alpha1/eval_rounds/round_{rd}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
                js = json.loads(j.read())
            print(round_sig(js['forget_truth_ratio'], r), end=' & ')
        except Exception:
            print("", end=' & ')

    print("", end='\\\\\n RT & ')

    for m in models:
        try:
            with open(f"./saves/TOFU/fedperm_only/retain_{f}_{m}-Instruct-lora_retain{rt}/eval_rounds/round_{rd2}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
                js = json.loads(j.read())
            print(round_sig(js['model_utility'], r), end=' & ')
        except Exception:
            print("", end=' & ')

    print("", end='\\\\\n RT & ')

    for m in models:
        try:
            with open(f"./saves/TOFU/fedperm_only/retain_{f}_{m}-Instruct-lora_retain{rt}/eval_rounds/round_{rd2}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
                js = json.loads(j.read())
            print(round_sig(js['forget_truth_ratio'], r), end=' & ')
        except Exception:
            print("", end=' & ')

    print("", end='\\\\\n FT & ')
    

Qwen2.5-1.5B & Qwen2.5-3B & Qwen2.5-7B & Llama-3.2-1B & Llama-3.2-3B & Llama-3.1-8B & \\
 FT & 0.525 & 0.61 & 0.651 & 0.571 & 0.636 & 0.693 & \\
 FT & 0.554 & 0.511 & 0.543 & 0.551 & 0.56 & 0.508 & \\
 SN & 0.522 & 0.579 & 0.632 & 0.552 & 0.615 & 0.67 & \\
 SN & 0.571 & 0.571 & 0.561 & 0.563 & 0.582 & 0.545 & \\
 RT & 0.521 & 0.605 & 0.616 & 0.563 & 0.605 & 0.669 & \\
 RT & 0.669 & 0.673 & 0.695 & 0.675 & 0.698 & 0.689 & \\
 FT & 0.518 & 0.598 & 0.651 & 0.578 & 0.635 & 0.666 & \\
 FT & 0.569 & 0.562 & 0.553 & 0.535 & 0.566 & 0.552 & \\
 SN & 0.518 & 0.583 & 0.633 & 0.571 & 0.632 & 0.655 & \\
 SN & 0.574 & 0.596 & 0.565 & 0.56 & 0.573 & 0.553 & \\
 RT & 0.524 & 0.603 & 0.66 & 0.554 & 0.605 & 0.656 & \\
 RT & 0.612 & 0.617 & 0.609 & 0.623 & 0.641 & 0.61 & \\
 FT & 0.531 & 0.612 & 0.619 & 0.566 & 0.644 & 0.661 & \\
 FT & 0.576 & 0.535 & 0.566 & 0.543 & 0.563 & 0.545 & \\
 SN & 0.525 & 0.592 & 0.643 & 0.518 & 0.625 & 0.625 & \\
 SN & 0.566 & 0.558 & 0.542 & 0.587 & 0.57 & 0.553 & \\
 RT & 

In [11]:
f = "FedAvg"
u = "SimNPO"

models = ["Qwen2.5-1.5B", "Qwen2.5-3B", "Qwen2.5-7B", "Llama-3.2-1B", "Llama-3.2-3B", "Llama-3.1-8B"]
split_ratios = ["90", "95", "99"]

# significant digit
r = 3

rd = 1
rd2 = 11
c = 1

def round_sig(x, sig):
    if x == 0 or str(x) == 'nan':
        return 0
    
    return round(x, sig-int(floor(log10(abs(x))))-1)

for m in models:
    print(m, end=' & ')

print("", end='\\\\\n FT & ')

for rt in split_ratios:
    for m in models:
        try:
            with open(f"./saves/TOFU/fedperm_only/full_{f}_{m}-Instruct-lora_retain{rt}/eval_rounds/round_{rd2}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
                js = json.loads(j.read())
            print(round_sig(js['model_utility'], r), end=' & ')
        except Exception:
            print("", end=' & ')

    print("", end='\\\\\n FT & ')

    for m in models:
        try:
            with open(f"./saves/TOFU/fedperm_only/full_{f}_{m}-Instruct-lora_retain{rt}/eval_rounds/round_{rd2}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
                js = json.loads(j.read())
            print(round_sig(js['forget_truth_ratio'], r), end=' & ')
        except Exception:
            print("", end=' & ')

    print("", end='\\\\\n GA & ')

    for m in models:
        try:
            with open(f"./saves/TOFU/fedperm_only/unlearn_{f}_{m}-Instruct-lora_retain{rt}_{u}_filter1_CE1_Alpha1/eval_rounds/round_{rd}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
                js = json.loads(j.read())
            print(round_sig(js['model_utility'], r), end=' & ')
        except Exception:
            print("", end=' & ')

    print("", end='\\\\\n GA & ')

    for m in models:
        try:
            with open(f"./saves/TOFU/fedperm_only/unlearn_{f}_{m}-Instruct-lora_retain{rt}_{u}_filter1_CE1_Alpha1/eval_rounds/round_{rd}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
                js = json.loads(j.read())
            print(round_sig(js['forget_truth_ratio'], r), end=' & ')
        except Exception:
            print("", end=' & ')

    print("", end='\\\\\n RT & ')

    for m in models:
        try:
            with open(f"./saves/TOFU/fedperm_only/retain_{f}_{m}-Instruct-lora_retain{rt}/eval_rounds/round_{rd2}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
                js = json.loads(j.read())
            print(round_sig(js['model_utility'], r), end=' & ')
        except Exception:
            print("", end=' & ')

    print("", end='\\\\\n RT & ')

    for m in models:
        try:
            with open(f"./saves/TOFU/fedperm_only/retain_{f}_{m}-Instruct-lora_retain{rt}/eval_rounds/round_{rd2}/client{c}_post_train/TOFU_SUMMARY.json", 'r') as j:
                js = json.loads(j.read())
            print(round_sig(js['forget_truth_ratio'], r), end=' & ')
        except Exception:
            print("", end=' & ')

    print("", end='\\\\\n FT & ')
    

Qwen2.5-1.5B & Qwen2.5-3B & Qwen2.5-7B & Llama-3.2-1B & Llama-3.2-3B & Llama-3.1-8B & \\
 FT &  &  &  &  &  &  & \\
 FT &  &  &  &  &  &  & \\
 GA &  &  &  &  &  &  & \\
 GA &  &  &  &  &  &  & \\
 RT &  &  &  &  &  &  & \\
 RT &  &  &  &  &  &  & \\
 FT &  &  &  &  &  &  & \\
 FT &  &  &  &  &  &  & \\
 GA &  &  &  &  &  &  & \\
 GA &  &  &  &  &  &  & \\
 RT &  &  &  &  &  &  & \\
 RT &  &  &  &  &  &  & \\
 FT &  &  &  &  &  &  & \\
 FT &  &  &  &  &  &  & \\
 GA &  &  &  &  &  &  & \\
 GA &  &  &  &  &  &  & \\
 RT &  &  &  &  &  &  & \\
 RT &  &  &  &  &  &  & \\
 FT & 

In [10]:
# 按轮次：在 source / target / authorized 三类上分别对 model_utility、forget_truth_ratio 取平均，
# 再对「每轮三类平均值」在轮次维度上宏平均。
# 拓扑来自 eval_rounds/round_*/participation.json（缺省时用 client_splits/fedperm_round_topology.json）。
# 目录与第二个 cell 对齐：同一 fed 算法下依次统计 full（微调）/ 各 unlearn / retain。

from math import floor, log10
import importlib

import pandas as pd

# Jupyter 会缓存 sys.modules 里的模块；若内核里先加载过旧版 partial_participation_eval，
# 不重启内核时直接 from ... import 新名字会失败，需 reload 后再导入。
import partial_participation_eval as _ppe
importlib.reload(_ppe)
from partial_participation_eval import aggregate_role_metrics_by_round

# ---------- 按需修改 ----------
fed_m = ["FedAdagrad"]  # "FedAdam", "FedAdagrad", ...
unlearn_m = ["GradAscent", "GradDiff", "NPO", "SimNPO"]
exp_mid = "Llama-3.2-1B-Instruct-lora_retain90"
unlearn_dir_suffix = "_filter1_CE1_Alpha1_Base1"

RD_FULL = 5
RD_UNLEARN = 2
RD_RETAIN = 5

# 评测子目录：per-client 为 "client" -> client{n}_post_train（与多数 TOFU_SUMMARY 路径一致）
CLIENT_EVAL_PREFIX = "client"

r = 3


def round_sig(x, sig):
    if x is None or x == 0 or str(x) == "nan":
        return 0
    return round(x, sig - int(floor(log10(abs(x)))) - 1)


def role_macro_payload(run_dir: str, max_round: int):
    return aggregate_role_metrics_by_round(
        run_dir=run_dir,
        metrics=["model_utility", "forget_truth_ratio"],  ## forget_truth_ratio
        max_round=max_round,
        stage="post_train",
        summary_file="TOFU_SUMMARY.json",
        client_eval_prefix=CLIENT_EVAL_PREFIX,
    )


def macro_row_dict(run_key: str, run_dir: str, max_round: int) -> dict:
    payload = role_macro_payload(run_dir, max_round)
    macro = payload["macro_mean_over_rounds"]
    row = {"run": run_key}
    for role in ("source", "target", "authorized"):
        row[f"{role}_MU"] = (macro.get(role) or {}).get("model_utility")
        row[f"{role}_FTR"] = (macro.get(role) or {}).get("forget_truth_ratio")  ## ## forget_truth_ratio
    return row


def latex_role_macro_row(label: str, run_dir: str, max_round: int) -> str:
    macro = role_macro_payload(run_dir, max_round)["macro_mean_over_rounds"]
    parts = [label]
    for role in ("source", "target", "authorized"):
        mu = (macro.get(role) or {}).get("model_utility")
        ftr = (macro.get(role) or {}).get("forget_truth_ratio")   ## forget_truth_ratio
        if mu is None or ftr is None:
            parts.extend(["", ""])
        else:
            parts.append(str(round_sig(mu, r)))
            parts.append(str(round_sig(ftr, r)))
    return " & ".join(parts) + " \\\\"


rows = []
latex_jobs = []
for f in fed_m:
    full_dir = f"./saves2/TOFU/fedperm_only/full_{f}_{exp_mid}"
    rows.append(macro_row_dict(f"full_{f}", full_dir, RD_FULL))
    latex_jobs.append((f"full_{f}", full_dir, RD_FULL))

    for u in unlearn_m:
        udir = (
            f"./saves2/TOFU/fedperm_only/"
            f"unlearn_{f}_{exp_mid}_{u}{unlearn_dir_suffix}"
        )
        rows.append(macro_row_dict(f"unlearn_{f}_{u}", udir, RD_UNLEARN))
        latex_jobs.append((f"unlearn_{f}_{u}", udir, RD_UNLEARN))

    retain_dir = f"./saves2/TOFU/fedperm_only/retain_{f}_{exp_mid}"
    rows.append(macro_row_dict(f"retain_{f}", retain_dir, RD_RETAIN))
    latex_jobs.append((f"retain_{f}", retain_dir, RD_RETAIN))

df = pd.DataFrame(rows)
display(df)

for label, path, max_r in latex_jobs:
    print(latex_role_macro_row(label, path, max_r))


,run,source_MU,source_FTR,target_MU,target_FTR,authorized_MU,authorized_FTR
0,full_FedAdagrad,0.381963,0.746783,0.365989,0.744850,0.367847,0.745462
1,unlearn_FedAdagrad_GradAscent,0.387204,0.741085,0.390390,0.742741,0.388816,0.743847
2,unlearn_FedAdagrad_GradDiff,0.387204,0.741085,0.390390,0.742741,0.388816,0.743847
3,unlearn_FedAdagrad_NPO,0.387204,0.741085,0.390097,0.742200,0.388816,0.743847
4,unlearn_FedAdagrad_SimNPO,0.387204,0.741085,0.390758,0.741519,0.388816,0.743847
5,retain_FedAdagrad,0.352094,0.745209,0.351541,0.745295,0.351640,0.745303


full_FedAdagrad & 0.382 & 0.747 & 0.366 & 0.745 & 0.368 & 0.745 \\
unlearn_FedAdagrad_GradAscent & 0.387 & 0.741 & 0.39 & 0.743 & 0.389 & 0.744 \\
unlearn_FedAdagrad_GradDiff & 0.387 & 0.741 & 0.39 & 0.743 & 0.389 & 0.744 \\
unlearn_FedAdagrad_NPO & 0.387 & 0.741 & 0.39 & 0.742 & 0.389 & 0.744 \\
unlearn_FedAdagrad_SimNPO & 0.387 & 0.741 & 0.391 & 0.742 & 0.389 & 0.744 \\
retain_FedAdagrad & 0.352 & 0.745 & 0.352 & 0.745 & 0.352 & 0.745 \\


In [11]:
# 按指定轮次：在 source / target / authorized 三类上分别对 model_utility、forget_truth_ratio 取平均
# 不同阶段单独指定轮次：full / unlearn / retain
#
# 这个 cell 不要求你先改 partial_participation_eval.py 的 aggregate_role_metrics_by_round。
# 它会在 notebook 内定义一个兼容版：支持 target_round，并复用模块里的底层工具函数。

from math import floor, log10
import importlib
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Union

import pandas as pd

import partial_participation_eval as _ppe
importlib.reload(_ppe)

# ---------- 按需修改 ----------
fed_m = ["FedAdagrad"]  # "FedAdam", "FedAdagrad", ...
unlearn_m = ["GradAscent", "GradDiff", "NPO", "SimNPO"]
exp_mid = "Llama-3.2-1B-Instruct-lora_retain90"
unlearn_dir_suffix = "_filter1_CE1_Alpha1_Base1"

# 分阶段单独指定轮次
TARGET_ROUND_FULL = 5
TARGET_ROUND_UNLEARN = 2
TARGET_ROUND_RETAIN = 5

# 评测子目录：per-client 为 "client" -> client{n}_post_train
CLIENT_EVAL_PREFIX = "client"

r = 3


def round_sig(x, sig):
    if x is None or x == 0 or str(x) == "nan":
        return 0
    return round(x, sig - int(floor(log10(abs(x)))) - 1)


# ========= notebook 内兼容版：支持 target_round =========
def aggregate_role_metrics_by_round_targetable(
    run_dir: Union[str, Path],
    metrics: Iterable[str],
    max_round: Optional[int] = None,
    target_round: Optional[int] = None,
    stage: str = "post_train",
    summary_file: str = "TOFU_SUMMARY.json",
    forget_source: Optional[int] = None,
    client_eval_prefix: str = "client",
) -> Dict[str, Any]:
    run_dir = Path(run_dir)
    metrics = list(metrics)
    st = _ppe._stage_name(stage, forget_source)
    rounds = _ppe._round_dirs(run_dir)

    if max_round is not None:
        rounds = [rd for rd in rounds if int(rd.name.split("_")[1]) <= int(max_round)]

    if target_round is not None:
        rounds = [rd for rd in rounds if int(rd.name.split("_")[1]) == int(target_round)]

    per_round: List[Dict[str, Any]] = []

    for round_dir in rounds:
        round_id = int(round_dir.name.split("_")[1])
        selected, st_map, src_list = _ppe._topology_for_round(run_dir, round_id)

        if not selected:
            round_means = {
                role: {m: None for m in metrics} for role in _ppe.ROLES_ORDER
            }
            per_round.append(
                {
                    "round_id": round_id,
                    "role_means": round_means,
                }
            )
            continue

        role_to_values: Dict[str, Dict[str, List[float]]] = {
            role: {m: [] for m in metrics} for role in _ppe.ROLES_ORDER
        }

        for client_idx in selected:
            role = _ppe._role_for_client(client_idx, selected, src_list, st_map)
            if role is None:
                continue

            subdir = _ppe._eval_subdir_name(client_idx, st, client_eval_prefix)
            summary_path = round_dir / subdir / summary_file
            summary = _ppe._load_json(summary_path)
            if not summary:
                continue

            for m in metrics:
                v = summary.get(m)
                if isinstance(v, (int, float)):
                    role_to_values[role][m].append(float(v))

        round_means: Dict[str, Dict[str, Optional[float]]] = {}
        for role in _ppe.ROLES_ORDER:
            round_means[role] = {}
            for m in metrics:
                vals = role_to_values[role][m]
                round_means[role][m] = sum(vals) / len(vals) if vals else None

        per_round.append(
            {
                "round_id": round_id,
                "role_means": round_means,
            }
        )

    macro_mean_over_rounds: Dict[str, Dict[str, Optional[float]]] = {}
    for role in _ppe.ROLES_ORDER:
        macro_mean_over_rounds[role] = {}
        for m in metrics:
            vals = [
                pr["role_means"][role][m]
                for pr in per_round
                if pr["role_means"][role][m] is not None
            ]
            macro_mean_over_rounds[role][m] = sum(vals) / len(vals) if vals else None

    result = {
        "run_dir": str(run_dir),
        "stage": st,
        "max_round": max_round,
        "target_round": target_round,
        "forget_source": forget_source,
        "client_eval_prefix": client_eval_prefix,
        "per_round_role_means": per_round,
        "macro_mean_over_rounds": macro_mean_over_rounds,
    }

    if target_round is not None:
        target_payload = next(
            (pr["role_means"] for pr in per_round if pr["round_id"] == int(target_round)),
            None,
        )
        result["target_round_role_means"] = target_payload

    return result


def role_target_round_payload(run_dir: str, target_round: int):
    return aggregate_role_metrics_by_round_targetable(
        run_dir=run_dir,
        metrics=["model_utility", "forget_truth_ratio"],
        target_round=target_round,
        stage="post_train",
        summary_file="TOFU_SUMMARY.json",
        client_eval_prefix=CLIENT_EVAL_PREFIX,
    )


def round_row_dict(run_key: str, run_dir: str, target_round: int) -> dict:
    payload = role_target_round_payload(run_dir, target_round)
    one_round = payload.get("target_round_role_means")

    row = {"run": run_key, "round": target_round}
    for role in ("source", "target", "authorized"):
        row[f"{role}_MU"] = (one_round.get(role) or {}).get("model_utility") if one_round else None
        row[f"{role}_FTR"] = (one_round.get(role) or {}).get("forget_truth_ratio") if one_round else None
    return row


def latex_role_round_row(label: str, run_dir: str, target_round: int) -> str:
    payload = role_target_round_payload(run_dir, target_round)
    one_round = payload.get("target_round_role_means")

    parts = [label]
    for role in ("source", "target", "authorized"):
        mu = (one_round.get(role) or {}).get("model_utility") if one_round else None
        ftr = (one_round.get(role) or {}).get("forget_truth_ratio") if one_round else None
        if mu is None or ftr is None:
            parts.extend(["", ""])
        else:
            parts.append(str(round_sig(mu, r)))
            parts.append(str(round_sig(ftr, r)))
    return " & ".join(parts) + " \\\\"


rows = []
latex_jobs = []

for f in fed_m:
    full_dir = f"./saves/TOFU/fedperm_only/full_{f}_{exp_mid}"
    rows.append(round_row_dict(f"full_{f}", full_dir, TARGET_ROUND_FULL))
    latex_jobs.append((f"full_{f}", full_dir, TARGET_ROUND_FULL))

    for u in unlearn_m:
        udir = (
            f"./saves/TOFU/fedperm_only/"
            f"unlearn_{f}_{exp_mid}_{u}{unlearn_dir_suffix}"
        )
        rows.append(round_row_dict(f"unlearn_{f}_{u}", udir, TARGET_ROUND_UNLEARN))
        latex_jobs.append((f"unlearn_{f}_{u}", udir, TARGET_ROUND_UNLEARN))

    retain_dir = f"./saves/TOFU/fedperm_only/retain_{f}_{exp_mid}"
    rows.append(round_row_dict(f"retain_{f}", retain_dir, TARGET_ROUND_RETAIN))
    latex_jobs.append((f"retain_{f}", retain_dir, TARGET_ROUND_RETAIN))

df = pd.DataFrame(rows)
display(df)

for label, path, round_id in latex_jobs:
    print(latex_role_round_row(label, path, round_id))

,run,round,source_MU,source_FTR,target_MU,target_FTR,authorized_MU,authorized_FTR
0,full_FedAdagrad,5,0.372055,0.611312,0.389570,0.687156,0.390979,0.717351
1,unlearn_FedAdagrad_GradAscent,2,NaN,NaN,NaN,NaN,NaN,NaN
2,unlearn_FedAdagrad_GradDiff,2,NaN,NaN,NaN,NaN,NaN,NaN
3,unlearn_FedAdagrad_NPO,2,NaN,NaN,NaN,NaN,NaN,NaN
4,unlearn_FedAdagrad_SimNPO,2,NaN,NaN,NaN,NaN,NaN,NaN
5,retain_FedAdagrad,5,0.377679,0.738882,0.400149,0.731327,0.386909,0.736478


full_FedAdagrad & 0.372 & 0.611 & 0.39 & 0.687 & 0.391 & 0.717 \\
unlearn_FedAdagrad_GradAscent & 0 & 0 &  &  & 0 & 0 \\
unlearn_FedAdagrad_GradDiff &  &  &  &  &  &  \\
unlearn_FedAdagrad_NPO &  &  &  &  &  &  \\
unlearn_FedAdagrad_SimNPO &  &  &  &  &  &  \\
retain_FedAdagrad & 0.378 & 0.739 & 0.4 & 0.731 & 0.387 & 0.736 \\
